In [4]:
from collections import deque
import copy

# Global counters
BACKTRACK_CALLS = 0
BACKTRACK_FAILURES = 0

def read_board(filename):
    file = open(filename, "r")
    lines = file.readlines()
    file.close()

    grid = []

    for line in lines:
        line = line.strip()
        row = []

        for ch in line:
            row.append(int(ch))

        grid.append(row)

    return grid

In [5]:
def print_board(grid):
    for i in range(9):
        for j in range(9):
            print(grid[i][j], end=" ")
        print()

In [6]:
def get_neighbors(row, col):
    neighbors = []

    # Row
    for j in range(9):
        if j != col:
            neighbors.append((row, j))

    # Column
    for i in range(9):
        if i != row:
            neighbors.append((i, col))

    # Box
    start_row = (row // 3) * 3
    start_col = (col // 3) * 3

    for i in range(start_row, start_row + 3):
        for j in range(start_col, start_col + 3):
            if i != row or j != col:
                if (i, j) not in neighbors:
                    neighbors.append((i, j))

    return neighbors

In [7]:
def initialize_domains(grid):
    domains = {}

    for i in range(9):
        for j in range(9):
            if grid[i][j] == 0:
                values = []
                for v in range(1, 10):
                    values.append(v)
                domains[(i, j)] = values
            else:
                domains[(i, j)] = [grid[i][j]]

    return domains

In [8]:
def ac3(domains):
    queue = deque()

    for cell in domains:
        neighbors = get_neighbors(cell[0], cell[1])
        for n in neighbors:
            queue.append((cell, n))

    while len(queue) > 0:
        (xi, xj) = queue.popleft()

        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False

            neighbors = get_neighbors(xi[0], xi[1])
            for n in neighbors:
                if n != xj:
                    queue.append((n, xi))

    return True

In [9]:
def revise(domains, xi, xj):
    revised = False

    to_remove = []

    for x in domains[xi]:
        found = False

        for y in domains[xj]:
            if x != y:
                found = True

        if not found:
            to_remove.append(x)

    for val in to_remove:
        domains[xi].remove(val)
        revised = True

    return revised

In [10]:
def forward_check(domains, cell, value):
    new_domains = copy.deepcopy(domains)

    neighbors = get_neighbors(cell[0], cell[1])

    for n in neighbors:
        if value in new_domains[n]:
            new_domains[n].remove(value)

            if len(new_domains[n]) == 0:
                return None

    return new_domains

In [11]:
def select_unassigned(domains):
    for cell in domains:
        if len(domains[cell]) > 1:
            return cell
    return None



In [12]:
def backtrack(domains):
    global BACKTRACK_CALLS
    global BACKTRACK_FAILURES

    BACKTRACK_CALLS += 1

    complete = True
    for cell in domains:
        if len(domains[cell]) != 1:
            complete = False
            break

    if complete:
        return domains

    var = select_unassigned(domains)

    for value in domains[var]:
        new_domains = copy.deepcopy(domains)
        new_domains[var] = [value]

        # Forward Checking
        new_domains = forward_check(new_domains, var, value)
        if new_domains is None:
            continue

        # AC-3
        if not ac3(new_domains):
            continue

        result = backtrack(new_domains)

        if result is not None:
            return result

    BACKTRACK_FAILURES += 1
    return None

In [13]:
def domains_to_grid(domains):
    grid = []

    for i in range(9):
        row = []
        for j in range(9):
            row.append(domains[(i, j)][0])
        grid.append(row)

    return grid

def solve(filename):
    global BACKTRACK_CALLS
    global BACKTRACK_FAILURES

    BACKTRACK_CALLS = 0
    BACKTRACK_FAILURES = 0

    grid = read_board(filename)
    domains = initialize_domains(grid)

    ac3(domains)

    result = backtrack(domains)

    if result is None:
        print("No solution")
        return

    solution = domains_to_grid(result)

    print("Solution:")
    print_board(solution)

    print("Backtrack calls:", BACKTRACK_CALLS)
    print("Backtrack failures:", BACKTRACK_FAILURES)

In [15]:
solve("easy.txt")
solve("medium.txt")
solve("hard.txt")
solve("veryhard.txt")

Solution:
7 8 4 9 3 2 1 5 6 
6 1 9 4 8 5 3 2 7 
2 3 5 1 7 6 4 8 9 
5 7 8 2 6 1 9 3 4 
3 4 1 8 9 7 5 6 2 
9 2 6 5 4 3 8 7 1 
4 5 3 7 2 9 6 1 8 
8 6 2 3 1 4 7 9 5 
1 9 7 6 5 8 2 4 3 
Backtrack calls: 1
Backtrack failures: 0
Solution:
8 7 5 9 3 6 1 4 2 
1 6 9 7 2 4 3 8 5 
2 4 3 8 5 1 6 7 9 
4 5 2 6 9 7 8 3 1 
9 8 6 4 1 3 2 5 7 
7 3 1 5 8 2 9 6 4 
5 1 7 3 6 9 4 2 8 
6 2 8 1 4 5 7 9 3 
3 9 4 2 7 8 5 1 6 
Backtrack calls: 3
Backtrack failures: 0
Solution:
1 5 2 3 4 6 8 9 7 
4 3 7 8 9 1 6 5 2 
6 8 9 5 7 2 3 1 4 
8 2 1 6 3 7 9 4 5 
5 4 3 1 8 9 7 2 6 
9 7 6 4 2 5 1 8 3 
7 9 8 2 5 3 4 6 1 
3 6 5 9 1 4 2 7 8 
2 1 4 7 6 8 5 3 9 
Backtrack calls: 9
Backtrack failures: 0
Solution:
4 3 1 8 6 7 9 2 5 
6 5 2 4 9 1 3 8 7 
8 9 7 5 3 2 1 6 4 
3 8 4 9 7 6 5 1 2 
5 1 9 2 8 4 7 3 6 
2 7 6 3 1 5 8 4 9 
9 4 3 7 2 8 6 5 1 
7 6 5 1 4 3 2 9 8 
1 2 8 6 5 9 4 7 3 
Backtrack calls: 68
Backtrack failures: 57
